# 08 -- MLP

Проверяю небольшую сеть над теми же признаками. Два скрытых слоя, без нового текстового encoder.
Все случайные операции используют seed 143.

In [1]:
from pathlib import Path
import sys
import json
import time
import random

import joblib
import numpy as np
import pandas as pd
import torch
from torch import nn
from sklearn.preprocessing import StandardScaler
from IPython.display import display

ml_root = Path.cwd() if (Path.cwd() / 'notebooks').exists() else Path.cwd().parent
import nbformat

# Подключаю только ячейки с функциями; расчеты других ноутбуков не запускаю.
for ml_file in ['02_feature_engineering.ipynb']:
    for ml_cell in nbformat.read(ml_root / 'notebooks' / ml_file, as_version=4).cells:
        if 'definitions' in ml_cell.metadata.get('tags', []):
            exec(ml_cell.source, globals())
ml_out = ml_root / 'outputs'
ml_seed = 143
random.seed(ml_seed)
np.random.seed(ml_seed)
torch.manual_seed(ml_seed)
torch.set_num_threads(4)
torch.use_deterministic_algorithms(True)

## Обучающие кандидаты

Группы без найденного позитива исключаю только из обучения. На dev они остаются в знаменателе.
Для классификаторов суммарный вес положительной и неразмеченной части каждого запроса одинаковый.
Каталожные ID и target не попадают в матрицу. Масштабирование обучаю только на rank_train.

In [2]:
ml_train = (
    pd.read_parquet(ml_out / 'cache/extended_rank_train.parquet')
    .sort_values(['query_id', 'pos'])
    .reset_index(drop=True)
)
ml_dev = (
    pd.read_parquet(ml_out / 'cache/extended_dev.parquet')
    .sort_values(['query_id', 'pos'])
    .reset_index(drop=True)
)
ml_queries = pd.read_parquet(ml_out / 'cache/eval_queries.parquet')
ml_dev_queries = ml_queries[ml_queries.split.eq('dev')]
ml_items = pd.read_parquet(ml_out / 'cache/items.parquet', columns=['item_id'])
ml_truth = json.loads((ml_out / 'cache/truth.json').read_text())
ml_dev_truth = {q: ml_truth[q] for q in ml_dev_queries.query_id}
ml_ids = ml_items.item_id.to_numpy()
ml_columns = feature_columns(ml_train)
ml_train['target'] = [
    int(ml_ids[p] in ml_truth[q]) for q, p in zip(ml_train.query_id, ml_train.pos)
]
ml_found = ml_train.groupby('query_id').target.transform('sum')
ml_train = ml_train[ml_found.gt(0)].reset_index(drop=True)
ml_npos = ml_train.groupby('query_id').target.transform('sum')
ml_n = ml_train.groupby('query_id').target.transform('size')
ml_weights = np.where(
    ml_train.target.eq(1), 0.5 / ml_npos, 0.5 / (ml_n - ml_npos).clip(lower=1)
)
ml_weights /= ml_weights.mean()
ml_x = model_matrix(ml_train, ml_columns)
ml_xdev = model_matrix(ml_dev, ml_columns)
ml_share = json.loads((ml_out / 'models/hybrid.json').read_text())['warm_share']
ml_results = []
ml_scores = ml_dev[['query_id', 'pos']].copy()
write_json(
    {'features': ml_columns, 'seed': ml_seed}, ml_out / 'models/extended_features.json'
)

In [3]:
def ml_evaluate(name, scores, seconds):
    predictions = predictions_from_frame(
        ml_dev.assign(score=scores), ml_items, score='score', k=100
    )
    per_query = recall_table(predictions, ml_dev_truth).merge(
        ml_dev_queries[['query_id', 'regime']], on='query_id'
    )
    values = per_query.groupby('regime')['recall@50'].mean()
    ml_scores[name] = scores
    ml_results.append(
        {
            'model': name,
            'cold': values['cold'],
            'warm': values['warm'],
            'objective': (1 - ml_share) * values['cold'] + ml_share * values['warm'],
            'fit_seconds': seconds,
        }
    )
    pd.DataFrame(ml_results).to_csv(ml_out / 'validation/models_ml.csv', index=False)
    per_query.to_parquet(ml_out / f'validation/dev_{name}.parquet', index=False)
    print(ml_results[-1], flush=True)

In [4]:
ml_scaler = StandardScaler().fit(ml_x)
ml_z = np.clip(ml_scaler.transform(ml_x), -10, 10).astype('float32')
ml_zdev = np.clip(ml_scaler.transform(ml_xdev), -10, 10).astype('float32')

## Обучение

Проверяю другой способ учитывать нелинейные сочетания признаков. Два скрытых слоя 64 и 32,
ReLU и dropout 0.1; AdamW, шесть эпох. Это нейросеть над поисковыми признаками, не новый текстовый encoder.
Seed 143 задаю для Python, NumPy и PyTorch; обучение выполняю детерминированно на CPU.

In [5]:
ml_mlp = nn.Sequential(
    nn.Linear(len(ml_columns), 64),
    nn.ReLU(),
    nn.Dropout(0.1),
    nn.Linear(64, 32),
    nn.ReLU(),
    nn.Linear(32, 1),
)
ml_optimizer = torch.optim.AdamW(ml_mlp.parameters(), lr=0.001, weight_decay=0.001)
ml_tensor = torch.from_numpy(ml_z)
ml_targets = torch.tensor(ml_train.target.to_numpy(), dtype=torch.float32)
ml_tensor_weights = torch.tensor(ml_weights, dtype=torch.float32)
ml_rng = np.random.default_rng(ml_seed)
ml_start = time.perf_counter()
ml_mlp_log = []

In [6]:
for ml_epoch in range(1, 7):
    ml_mlp.train()
    ml_order = ml_rng.permutation(len(ml_train))
    ml_loss_sum = 0.0
    for ml_begin in range(0, len(ml_order), 4096):
        ml_idx = ml_order[ml_begin : ml_begin + 4096]
        ml_optimizer.zero_grad()
        ml_logits = ml_mlp(ml_tensor[ml_idx]).squeeze(1)
        ml_losses = nn.functional.binary_cross_entropy_with_logits(
            ml_logits, ml_targets[ml_idx], reduction='none'
        )
        ml_loss = (ml_losses * ml_tensor_weights[ml_idx]).mean()
        ml_loss.backward()
        ml_optimizer.step()
        ml_loss_sum += float(ml_loss.detach()) * len(ml_idx)
    ml_mlp_log.append({'epoch': ml_epoch, 'weighted_loss': ml_loss_sum / len(ml_train)})
    print(ml_mlp_log[-1], flush=True)
    if ml_epoch in [3, 6]:
        ml_mlp.eval()
        with torch.no_grad():
            ml_prediction = np.concatenate(
                [
                    ml_mlp(torch.from_numpy(ml_zdev[i : i + 8192])).squeeze(1).numpy()
                    for i in range(0, len(ml_zdev), 8192)
                ]
            )
        ml_evaluate(f'mlp_{ml_epoch}', ml_prediction, time.perf_counter() - ml_start)
        torch.save(ml_mlp.state_dict(), ml_out / f'models/mlp_{ml_epoch}.pt')
pd.DataFrame(ml_mlp_log).to_csv(ml_out / 'validation/mlp_training.csv', index=False)

{'epoch': 1, 'weighted_loss': 0.290314812053155}


{'epoch': 2, 'weighted_loss': 0.23318034933149537}


{'epoch': 3, 'weighted_loss': 0.2231079072421921}


{'model': 'mlp_3', 'cold': np.float64(0.7954722222222221), 'warm': np.float64(0.8053452380952381), 'objective': np.float64(0.7991725899168802), 'fit_seconds': 2.189644957999917}


{'epoch': 4, 'weighted_loss': 0.21831444378514198}


{'epoch': 5, 'weighted_loss': 0.21198874233351941}


{'epoch': 6, 'weighted_loss': 0.2083919415719406}


{'model': 'mlp_6', 'cold': np.float64(0.7969722222222222), 'warm': np.float64(0.800154761904762), 'objective': np.float64(0.7981650256350501), 'fit_seconds': 4.798435165999763}


In [7]:
ml_scores.to_parquet(ml_out / 'cache/scores_ml.parquet', index=False)
pd.DataFrame(ml_results).to_csv(ml_out / 'validation/models_ml.csv', index=False)
display(pd.DataFrame(ml_results))

,model,cold,warm,objective,fit_seconds
0,mlp_3,0.795472,0.805345,0.799173,2.189645
1,mlp_6,0.796972,0.800155,0.798165,4.798435


## Вывод

MLP дала 0.79917 после трех эпох и 0.79817 после шести. Более долгое обучение не помогло.
Для сравнения ансамблей беру вариант на трех эпохах, выбранный по dev. CatBoost пока сильнее.